# RAG, Embeddings & Vector Databases — Module 04 (Day 16–Day 20)
**Level:** High | **Google Colab / Jupyter**

Objectives: understand hallucinations and RAG; learn embeddings, semantic similarity, chunking and vector databases; retrieve context for document Q&A; build Chat With Your Documents.

**Pipeline:** Document → Chunking → Embeddings → Vector DB → Retrieval → LLM → Answer.

## 1. Install libraries

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu numpy openai

## 2. Hallucinations and RAG

An LLM hallucination is a fluent but false or unsupported answer. Retrieval-Augmented Generation (RAG) retrieves relevant passages from an external knowledge base and supplies them to the model as context. RAG can improve grounding, but it does not guarantee correctness.

## 3. Create a sample document collection

In [ ]:
documents = [
 {"source":"rag_intro.txt","text":"Retrieval-Augmented Generation combines information retrieval with text generation. A retriever finds relevant passages from a knowledge base and provides them to a language model as context."},
 {"source":"embeddings.txt","text":"An embedding is a numerical vector representation of text. Texts with related meanings often have vectors close in embedding space. Embeddings are useful for semantic search."},
 {"source":"chunking.txt","text":"Chunking splits long documents into smaller text sections. Overlap keeps some words from the previous chunk so information near chunk boundaries is less likely to be lost."},
 {"source":"vector_db.txt","text":"A vector database stores vector embeddings and supports similarity search. FAISS is a library for efficient similarity search and clustering of dense vectors."},
 {"source":"limitations.txt","text":"RAG can improve grounding but cannot guarantee factual answers. Poor chunking, irrelevant retrieval, missing documents and ambiguous questions can reduce answer quality. Check answers against source documents."}
]
print("Sample documents:",len(documents))

## 4. Text chunking

In [ ]:
def chunk_text(text, chunk_size=45, overlap=10):
    words=text.split()
    result=[]
    start=0
    while start < len(words):
        end=min(start+chunk_size,len(words))
        result.append(" ".join(words[start:end]))
        if end==len(words): break
        start=end-overlap
    return result

chunks=[]
sources=[]
for doc in documents:
    parts=chunk_text(doc["text"],45,10)
    chunks.extend(parts)
    sources.extend([doc["source"]]*len(parts))
for i,c in enumerate(chunks):
    print(f"{i+1}. {sources[i]}: {c}\n")

## 5. Generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings=embedding_model.encode(chunks,convert_to_numpy=True,normalize_embeddings=True)
embeddings=np.asarray(embeddings,dtype="float32")
print("Embedding matrix shape:",embeddings.shape)

## 6. Semantic similarity

For normalized vectors, dot product equals cosine similarity. Higher scores generally indicate closer semantic meaning. Similarity scores rank results; they are not probabilities of correctness.

In [ ]:
query="How does a system find relevant information before an AI answers?"
q=embedding_model.encode([query],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
scores=embeddings @ q[0]
for idx in np.argsort(scores)[::-1][:3]:
    print(f"{scores[idx]:.3f} | {sources[idx]} | {chunks[idx]}")

## 7. Build a FAISS vector index

In [ ]:
import faiss
index=faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS index created. Vectors stored:",index.ntotal)

## 8. Retrieve relevant chunks

In [ ]:
def retrieve(question,top_k=3):
    q=embedding_model.encode([question],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores,ids=index.search(q,min(top_k,len(chunks)))
    return [{"score":float(s),"source":sources[int(i)],"text":chunks[int(i)]}
            for s,i in zip(scores[0],ids[0]) if i>=0]

question="What are embeddings and why are they used for semantic search?"
retrieved=retrieve(question)
for item in retrieved:
    print(item["source"],round(item["score"],3),item["text"],"\n")

## 9. Optional: connect retrieved context to an LLM API

The retrieval steps above work without an API key. To generate a natural-language answer, add your own OPENAI_API_KEY in Colab Secrets. API use may incur charges.

In [ ]:
from openai import OpenAI
from google.colab import userdata

api_key=userdata.get("OPENAI_API_KEY")
if api_key:
    client=OpenAI(api_key=api_key)
    context="\n\n".join(f"Source: {x['source']}\n{x['text']}" for x in retrieved)
    response=client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"system","content":"Answer only from the supplied context. If the answer is absent, say you cannot find it. Do not invent facts."},
            {"role":"user","content":f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0.2,max_tokens=250
    )
    print(response.choices[0].message.content)
    print("Token usage:",response.usage)
else:
    print("OPENAI_API_KEY not found. Retrieval is complete; add a key for LLM answer generation.")

## 10. Evaluate retrieval

In [ ]:
questions=[
 "What does RAG mean?",
 "Why split a document into chunks?",
 "What is FAISS used for?",
 "Can RAG guarantee every answer is correct?"
]
for q in questions:
    print("\nQUESTION:",q)
    for item in retrieve(q,2):
        print("-",item["source"],f"({item['score']:.3f})",item["text"])

## 11. Chunk-size experiment
Change `chunk_size` and `overlap` in the chunking cell, rebuild embeddings and the FAISS index, then compare retrieved passages.

Record:
- Chunk size:
- Overlap:
- Test question:
- Most relevant source:
- Did context quality improve? Why?

## 12. Final application: Chat With Your Documents
The GitHub repository contains a Streamlit app that accepts PDF, DOCX and TXT files, extracts text, chunks it, generates embeddings, stores vectors in FAISS, retrieves relevant passages and sends context to an LLM for answer generation. Run locally with `streamlit run app.py`.

## 13. Limitations and responsible use
Scanned PDFs may require OCR. Retrieval quality depends on document quality, chunking and embeddings. Similarity scores are not confidence probabilities. RAG can still miss relevant passages or hallucinate. Verify answers and do not upload sensitive documents to unapproved services.

## 14. Conclusion
I learned how RAG can ground LLM answers in external documents. I processed text into chunks, generated embeddings, compared semantic similarity, stored vectors in FAISS, retrieved relevant passages and connected retrieved context to an LLM. I also prepared a Streamlit document Q&A application.

**Submission:** Run the notebook, review retrieval outputs, complete the chunk-size experiment and upload the GitHub repository.